In [ ]:
"""
Notebook 2: Phase 2 Dual-Stream Training Pipeline
Requires dna_promoter_cache.npz generated by Notebook 1.
Run this on a Kaggle GPU instance.
"""

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import pearsonr
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
import matplotlib.pyplot as plt

# --- PATHS & CONSTANTS ---
# REPO_ROOT should point to the main dataset containing your parquet files and gene coords
REPO_ROOT = Path("/kaggle/input/datasets/ericdu847/dna-promoter-cache") 

OUTPUT_DIR = Path("/kaggle/working")
RESULTS_DIR = OUTPUT_DIR / "results"
FIGURES_DIR = OUTPUT_DIR / "figures"

# The newly discovered path for your Notebook 1 output!
DNA_CACHE_PATH = Path("/kaggle/input/datasets/ericdu847/dna-promoter-cache/dna_promoter_cache.npz")

GROUPS = ["B", "monocyte", "CD8T", "naive_CD4T"]
SEQ_LEN = 2000

FALLBACK_TF_LIST = [
    "STAT1", "STAT2", "STAT3", "STAT4", "STAT5A", "STAT5B", "STAT6", "IRF1", "IRF3", "IRF4",
    "IRF7", "IRF8", "NFKB1", "NFKB2", "RELA", "RELB", "REL", "JUN", "JUNB", "JUND", "FOS",
    "FOSB", "FOSL1", "FOSL2", "ATF3", "ATF4", "NR4A1", "NR4A2", "NR4A3", "EGR1", "EGR2",
    "EGR3", "MYC", "MYB", "GATA1", "GATA2", "GATA3", "TBX21", "RORC", "FOXP3", "SPI1",
    "CEBPA", "CEBPB", "CEBPD", "IKZF1", "IKZF2", "IKZF3", "BCL6", "PRDM1", "BATF", "BATF3",
    "EOMES", "RUNX1", "RUNX3", "TCF7", "LEF1", "ID2", "ID3", "KLF2", "KLF4", "KLF6",
    "SOCS1", "SOCS3", "PPARG", "XBP1", "ETS1", "ETS2", "ELF1", "ELK1", "MAF", "MAFB",
]

# --- DATA PREP ---
class SingleCellDataset(Dataset):
    def __init__(self, samples_df, dna_dict, tf_dict, num_tfs):
        self.samples = samples_df
        self.dna_dict = dna_dict
        self.tf_dict = tf_dict
        self.num_tfs = num_tfs

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        row = self.samples.iloc[idx]
        x_cis = torch.tensor(self.dna_dict.get(row["gene_id"], np.zeros((SEQ_LEN, 4))), dtype=torch.float32)
        x_trans = torch.tensor(self.tf_dict.get(row["condition_id"], np.zeros(self.num_tfs)), dtype=torch.float32)
        y = torch.tensor([row["target_delta_expression"]], dtype=torch.float32)
        return x_cis, x_trans, y

def load_data_and_tfs(group):
    df = pd.read_parquet(REPO_ROOT / f"scbloodnl_move1_delta_{group}.parquet").fillna(0.0)
    tf_present = [g for g in FALLBACK_TF_LIST if g in df.index]
    tf_block = df.loc[tf_present]
    
    tf_dict = {}
    for cond in df.columns:
        vec = np.zeros(len(FALLBACK_TF_LIST), dtype=np.float32)
        vec[:len(tf_present)] = tf_block[cond].values
        tf_dict[cond] = vec

    samples = df.reset_index().melt(id_vars="index", var_name="condition_id", value_name="target_delta_expression").rename(columns={"index": "gene_id"})
    return samples, tf_dict, len(FALLBACK_TF_LIST)

# --- MODEL ---
class CompositionalModel(nn.Module):
    def __init__(self, num_tfs=71):
        super().__init__()
        self.cis = nn.Sequential(
            nn.Conv1d(4, 64, 15, padding=7), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(64, 128, 9, padding=4), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(128, 256, 5, padding=2), nn.BatchNorm1d(256), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
            nn.Flatten(), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2)
        )
        self.trans = nn.Sequential(
            nn.Linear(num_tfs, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.1)
        )
        self.fusion = nn.Sequential(
            nn.Linear(192, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1)
        )

    def forward(self, x_cis, x_trans):
        x_cis = x_cis.transpose(1, 2) if x_cis.shape[1] != 4 else x_cis
        return self.fusion(torch.cat([self.cis(x_cis), self.trans(x_trans)], dim=1))

# --- TRAINING LOOP ---
def run_fold(group, held_out, dna_dict, device="cuda"):
    samples, tf_dict, num_tfs = load_data_and_tfs(group)
    train_ds = SingleCellDataset(samples[samples["condition_id"] != held_out], dna_dict, tf_dict, num_tfs)
    test_ds = SingleCellDataset(samples[samples["condition_id"] == held_out], dna_dict, tf_dict, num_tfs)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_ds, batch_size=32)

    model = CompositionalModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
    criterion = nn.HuberLoss()

    losses = []
    for epoch in range(15):
        model.train()
        epoch_loss = 0
        for x_c, x_t, y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(x_c.to(device), x_t.to(device)), y.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(train_loader))

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x_c, x_t, y in test_loader:
            preds.extend(model(x_c.to(device), x_t.to(device)).cpu().numpy().flatten())
            targets.extend(y.numpy().flatten())

    r, p = pearsonr(preds, targets) if np.std(preds) > 1e-9 else (0.0, 1.0)
    
    # Save Loss Plot
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    plt.figure()
    plt.plot(range(1, 16), losses, marker="o")
    plt.title(f"{group} - {held_out} Loss")
    plt.savefig(FIGURES_DIR / f"loss_{group}_{held_out}.png")
    plt.close()

    return {"group": group, "held_out": held_out, "r": r, "p": p, "final_loss": losses[-1]}

# --- EXECUTION ---
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Loading DNA cache...")
with np.load(DNA_CACHE_PATH) as npz:
    dna_dict = {k: npz[k] for k in npz.files}

results = []
for grp in GROUPS:
    for cond in pd.read_parquet(REPO_ROOT / f"scbloodnl_move1_delta_{grp}.parquet").columns:
        print(f"Training {grp} holding out {cond}...")
        res = run_fold(grp, cond, dna_dict)
        print(f"  -> Pearson r: {res['r']:.4f}")
        results.append(res)

pd.DataFrame(results).to_csv(RESULTS_DIR / "final_results.csv", index=False)
print("All folds complete!")

In [ ]:
import os
print("Looking for files in /kaggle/input/...")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if "dna_promoter" in filename or "cache" in filename:
            print("FOUND IT:", os.path.join(dirname, filename))